# 📡 TRAI India Telecom Market Analysis
## Wireless Subscriber Trends, Market Share & Broadband Intelligence (Jan 2025 – Aug 2026)

**Author:** Harshwardhan Patil  
**Dataset:** TRAI Telecom Analytics Dataset (`TRAI_Telecom_Analytics_Dataset.xlsx`, `TRAI_Wireless_LSA_Long.csv`)  
**Source:** Telecom Regulatory Authority of India (TRAI) Press Releases  

---
### Project Objectives
1. Clean and validate TRAI's raw telecom subscriber data
2. Analyse national wireless subscriber growth trends
3. Track operator market share evolution over 19 months
4. Perform LSA (Licensed Service Area) level regional analysis
5. Examine broadband penetration and urban/rural digital divide
6. Study Mobile Number Portability (MNP) patterns
7. Derive competitive intelligence insights from the data

---
## 1. Setup & Library Imports

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Colour palette per operator
OPERATOR_COLORS = {
    'Reliance Jio':          '#2e7d32',
    'Bharti Airtel':         '#e65100',
    'Vodafone Idea':         '#1565c0',
    'BSNL':                  '#880e4f',
    'MTNL':                  '#795548',
    'Reliance Communications': '#607d8b',
}

print('✅ Libraries loaded successfully')

---
## 2. Data Loading

In [ ]:
# ── Load CSV: LSA-level wireless subscriber data ──────────────────────────────
lsa_raw = pd.read_csv('TRAI_Wireless_LSA_Long.csv', parse_dates=['month'])

# ── Load Excel sheets ─────────────────────────────────────────────────────────
national_summary   = pd.read_excel('TRAI_Telecom_Analytics_Dataset.xlsx',
                                   sheet_name='National_Summary',   parse_dates=['month'])
nat_op_monthly     = pd.read_excel('TRAI_Telecom_Analytics_Dataset.xlsx',
                                   sheet_name='National_Operator_Monthly', parse_dates=['month'])
lsa_totals         = pd.read_excel('TRAI_Telecom_Analytics_Dataset.xlsx',
                                   sheet_name='LSA_Totals',         parse_dates=['month'])
broadband_top5     = pd.read_excel('TRAI_Telecom_Analytics_Dataset.xlsx',
                                   sheet_name='Broadband_Top5',     parse_dates=['month'])

print(f'LSA raw rows      : {len(lsa_raw):,}')
print(f'National summary  : {len(national_summary)} rows')
print(f'Nat. op. monthly  : {len(nat_op_monthly)} rows')
print(f'LSA totals        : {len(lsa_totals)} rows')
print(f'Broadband top-5   : {len(broadband_top5)} rows')

In [ ]:
# Preview LSA data
lsa_raw.head(8)

In [ ]:
# Preview national operator monthly
nat_op_monthly[['month','operator','wireless_mobile_subscribers',
                'market_share_pct','month_over_month_change',
                'month_over_month_growth_pct']].head(12)

---
## 3. Data Cleaning & Validation

In [ ]:
# ── 3.1 LSA Data Quality Audit ───────────────────────────────────────────────
print('=== LSA RAW DATA QUALITY REPORT ===')
print(f'Total rows          : {len(lsa_raw):,}')
print(f'Distinct months     : {lsa_raw["month"].nunique()} → {sorted(lsa_raw["month"].dt.strftime("%b %Y").unique())}')
print(f'Distinct LSAs       : {lsa_raw["lsa"].nunique()}')
print(f'Distinct operators  : {lsa_raw["operator"].nunique()}')
print()
print('── value_status breakdown ──')
print(lsa_raw['value_status'].value_counts())
print()
print('── Missing subscriber values by operator ──')
print(lsa_raw[lsa_raw['value_status']=='blank_in_source']['operator'].value_counts())
print()
print(f'Duplicate rows (month+lsa+operator): {lsa_raw.duplicated(["month","lsa","operator"]).sum()}')
print(f'Negative subscriber counts         : {(lsa_raw["wireless_mobile_subscribers"] < 0).sum()}')

In [ ]:
# ── 3.2 Clean LSA Dataset ────────────────────────────────────────────────────
# Keep only rows with actual reported values (exclude blank_in_source)
lsa_clean = lsa_raw[lsa_raw['value_status'] == 'reported'].copy()

# Standardise LSA names (minor whitespace/alias fixes found in raw)
lsa_name_map = {
    'U.P.(W )': 'Uttar Pradesh West',
}
lsa_clean['lsa'] = lsa_clean['lsa'].replace(lsa_name_map)

# Convert subscribers to integer
lsa_clean['wireless_mobile_subscribers'] = (
    lsa_clean['wireless_mobile_subscribers'].astype(float).astype('Int64')
)

# Add millions column for readability
lsa_clean['subs_millions'] = lsa_clean['wireless_mobile_subscribers'] / 1e6

# Sort
lsa_clean = lsa_clean.sort_values(['month', 'lsa', 'operator']).reset_index(drop=True)

print(f'Clean LSA rows : {len(lsa_clean):,} (from {len(lsa_raw):,} raw)')
lsa_clean.dtypes

In [ ]:
# ── 3.3 Clean National Operator Monthly ──────────────────────────────────────
nom = nat_op_monthly.copy()
nom['subs_millions'] = nom['wireless_mobile_subscribers'] / 1e6
nom = nom.sort_values(['month','operator']).reset_index(drop=True)

# Flag major operators only (exclude near-defunct ones for main charts)
MAJOR_OPS = ['Reliance Jio', 'Bharti Airtel', 'Vodafone Idea', 'BSNL']
nom_major = nom[nom['operator'].isin(MAJOR_OPS)].copy()

print('National operator monthly — shape:', nom.shape)
print('Months available:', sorted(nom['month'].dt.strftime('%b %Y').unique()))

In [ ]:
# ── 3.4 Clean National Summary ───────────────────────────────────────────────
ns = national_summary.sort_values('month').reset_index(drop=True)

# ── 3.5 Clean Broadband ──────────────────────────────────────────────────────
bb = broadband_top5.sort_values(['month','operator']).reset_index(drop=True)

# ── 3.6 Missing months summary ───────────────────────────────────────────────
all_months = pd.date_range('2025-01-01', '2026-08-01', freq='MS')
available  = nom['month'].unique()
missing    = [m for m in all_months if m not in available]
print(f'Missing months in national operator data ({len(missing)}):')
for m in missing:
    print(' ', m.strftime('%b %Y'))

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ── 4.1 Statistical Summary ───────────────────────────────────────────────────
print('=== LSA SUBSCRIBER STATISTICS (reported values only, in Millions) ===')
lsa_clean.groupby('operator')['subs_millions'].describe().round(2)

In [ ]:
# ── 4.2 National total subscribers per month ─────────────────────────────────
monthly_total = (
    nom.groupby('month')['subs_millions'].sum()
    .reset_index(name='total_subs_millions')
)
monthly_total

In [ ]:
# ── 4.3 Market share summary (latest month) ──────────────────────────────────
latest_month = nom['month'].max()
latest = nom[nom['month'] == latest_month][['operator','subs_millions','market_share_pct']]
latest = latest.sort_values('market_share_pct', ascending=False).reset_index(drop=True)
print(f'Latest available month: {latest_month.strftime("%B %Y")}')
latest

In [ ]:
# ── 4.4 LSA-level totals for latest month ────────────────────────────────────
lsa_jan25 = lsa_clean[lsa_clean['month'] == '2025-01-01']
lsa_totals_jan25 = (
    lsa_jan25.groupby('lsa')['subs_millions']
    .sum().sort_values(ascending=False)
    .reset_index(name='total_subs_millions')
)
print('Top 10 LSAs by total subscribers (Jan 2025):')
lsa_totals_jan25.head(10)

---
## 5. Visualisations
### 5.1 National Subscriber Growth Trend

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.fill_between(monthly_total['month'], monthly_total['total_subs_millions'],
                alpha=0.12, color='steelblue')
ax.plot(monthly_total['month'], monthly_total['total_subs_millions'],
        marker='o', color='steelblue', linewidth=2.5, markersize=6, label='Total Subscribers')

# Annotate first and last
for idx in [0, len(monthly_total)-1]:
    row = monthly_total.iloc[idx]
    ax.annotate(f"{row['total_subs_millions']:.0f}M",
                xy=(row['month'], row['total_subs_millions']),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=9, fontweight='bold', color='steelblue')

# Data gap shading
gap_start = pd.Timestamp('2025-08-01')
gap_end   = pd.Timestamp('2026-02-01')
ax.axvspan(gap_start, gap_end, alpha=0.07, color='orange', label='Data gap')
ax.text(pd.Timestamp('2025-10-15'), monthly_total['total_subs_millions'].max() * 0.97,
        'Data Gap\n(Sep 25–Jan 26)', ha='center', fontsize=8, color='darkorange')

ax.set_title('All-India Wireless Subscriber Base (Jan 2025 – Aug 2026)',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Month')
ax.set_ylabel('Subscribers (Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}M'))
ax.legend()
fig.tight_layout()
plt.savefig('fig1_national_growth.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Operator Subscriber Trends

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

for op in MAJOR_OPS:
    df_op = nom_major[nom_major['operator'] == op].sort_values('month')
    ax.plot(df_op['month'], df_op['subs_millions'],
            marker='o', linewidth=2.2, markersize=5,
            label=op, color=OPERATOR_COLORS[op])

ax.set_title('Wireless Subscribers by Operator (Millions)', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Month')
ax.set_ylabel('Subscribers (Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}M'))
ax.legend(loc='upper left', fontsize=10)
fig.tight_layout()
plt.savefig('fig2_operator_trends.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Market Share Evolution — Stacked Area Chart

In [ ]:
# Pivot for stacked area
ms_pivot = (
    nom_major.pivot_table(index='month', columns='operator',
                          values='market_share_pct', aggfunc='first')
    .sort_index()
)

fig, ax = plt.subplots(figsize=(13, 5))
cols_order = ['Reliance Jio', 'Bharti Airtel', 'Vodafone Idea', 'BSNL']
colors     = [OPERATOR_COLORS[c] for c in cols_order]

ax.stackplot(ms_pivot.index,
             [ms_pivot[c] for c in cols_order],
             labels=cols_order, colors=colors, alpha=0.82)

ax.set_title('Wireless Market Share Evolution — Stacked Area (%)', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Month')
ax.set_ylabel('Market Share (%)')
ax.set_ylim(0, 100)
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
fig.tight_layout()
plt.savefig('fig3_market_share_stacked.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Market Share Pie Chart — Latest Month

In [ ]:
latest_all = nom[nom['month'] == latest_month].copy()
latest_all = latest_all.sort_values('market_share_pct', ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
wedge_colors = [OPERATOR_COLORS.get(op, '#999') for op in latest_all['operator']]
wedges, texts, autotexts = ax.pie(
    latest_all['market_share_pct'],
    labels=latest_all['operator'],
    autopct='%1.1f%%',
    colors=wedge_colors,
    startangle=140,
    pctdistance=0.82,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for t in autotexts:
    t.set_fontsize(11)
    t.set_fontweight('bold')

ax.set_title(f'Wireless Market Share — {latest_month.strftime("%B %Y")}',
             fontsize=14, fontweight='bold', pad=16)
fig.tight_layout()
plt.savefig('fig4_market_share_pie.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Month-over-Month Net Additions — Bar Chart

In [ ]:
mom = nom_major.dropna(subset=['month_over_month_change']).copy()
mom['mom_millions'] = mom['month_over_month_change'] / 1e6

fig, ax = plt.subplots(figsize=(14, 5))

width = 18  # days width per bar in matplotlib date units
offsets = {'Reliance Jio': -27, 'Bharti Airtel': -9, 'Vodafone Idea': 9, 'BSNL': 27}

for op in MAJOR_OPS:
    df_op = mom[mom['operator'] == op]
    x_vals = [d + pd.Timedelta(days=offsets[op]) for d in df_op['month']]
    bar_colors = [OPERATOR_COLORS[op] if v >= 0 else '#ccc' for v in df_op['mom_millions']]
    ax.bar(x_vals, df_op['mom_millions'], width=width,
           color=bar_colors, label=op, alpha=0.9)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Month-over-Month Subscriber Change by Operator (Millions)',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Month')
ax.set_ylabel('Net Additions (Millions)')
ax.legend(fontsize=10)
fig.tight_layout()
plt.savefig('fig5_mom_additions.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6 Jio vs Airtel Gap — Closing the Distance

In [ ]:
jio_data    = nom[nom['operator'] == 'Reliance Jio'].set_index('month')['subs_millions']
airtel_data = nom[nom['operator'] == 'Bharti Airtel'].set_index('month')['subs_millions']
gap = (jio_data - airtel_data).dropna().reset_index()
gap.columns = ['month', 'gap_millions']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: absolute subscribers
for op, color in [('Reliance Jio', '#2e7d32'), ('Bharti Airtel', '#e65100')]:
    d = nom[nom['operator'] == op].sort_values('month')
    axes[0].plot(d['month'], d['subs_millions'], marker='o', linewidth=2.5,
                 markersize=5, label=op, color=color)
axes[0].set_title('Jio vs Airtel — Absolute Subscribers (M)', fontweight='bold')
axes[0].set_ylabel('Subscribers (Millions)')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))

# Right: gap
axes[1].fill_between(gap['month'], gap['gap_millions'], alpha=0.2, color='purple')
axes[1].plot(gap['month'], gap['gap_millions'], color='purple', linewidth=2.5, marker='o')
axes[1].axhline(0, color='red', linewidth=1, linestyle='--', alpha=0.6)
axes[1].set_title('Jio–Airtel Subscriber Gap (Millions)', fontweight='bold')
axes[1].set_ylabel('Gap (Millions)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))

fig.suptitle('Jio vs Airtel: Convergence Story', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.savefig('fig6_jio_airtel_gap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Gap Jan 2025 : {gap[gap["month"]=="2025-01-01"]["gap_millions"].values[0]:.1f}M')
print(f'Gap Aug 2026 : {gap[gap["month"]=="2026-08-01"]["gap_millions"].values[0]:.1f}M')

### 5.7 LSA-Level Subscriber Heatmap

In [ ]:
# Pivot: LSA vs month — total subscribers across all operators
lsa_heat = (
    lsa_clean.groupby(['lsa', 'month'])['subs_millions']
    .sum()
    .unstack('month')
)
lsa_heat.columns = lsa_heat.columns.strftime('%b\'%y')
lsa_heat = lsa_heat.sort_values(lsa_heat.columns[-1], ascending=False)

fig, ax = plt.subplots(figsize=(16, 9))
sns.heatmap(lsa_heat, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Subscribers (Millions)'},
            annot_kws={'size': 8})
ax.set_title('Total Wireless Subscribers by LSA and Month (Millions)',
             fontsize=14, fontweight='bold', pad=16)
ax.set_xlabel('Month')
ax.set_ylabel('Licensed Service Area')
fig.tight_layout()
plt.savefig('fig7_lsa_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.8 Operator Share by LSA — Grouped Bar (Jan 2025)

In [ ]:
lsa_j25 = lsa_clean[lsa_clean['month'] == '2025-01-01'].copy()
lsa_j25 = lsa_j25[lsa_j25['operator'].isin(MAJOR_OPS)]

# Calculate share within each LSA
lsa_j25['lsa_total'] = lsa_j25.groupby('lsa')['subs_millions'].transform('sum')
lsa_j25['share'] = lsa_j25['subs_millions'] / lsa_j25['lsa_total'] * 100

pivot_share = lsa_j25.pivot_table(index='lsa', columns='operator',
                                   values='share', fill_value=0)
# Sort by Jio share desc
if 'Reliance Jio' in pivot_share.columns:
    pivot_share = pivot_share.sort_values('Reliance Jio', ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
pivot_share[MAJOR_OPS].plot(
    kind='barh', stacked=True, ax=ax,
    color=[OPERATOR_COLORS[o] for o in MAJOR_OPS],
    alpha=0.88
)
ax.set_title('Operator Market Share by LSA — Jan 2025 (%)',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Market Share (%)')
ax.set_ylabel('')
ax.set_xlim(0, 100)
ax.legend(loc='lower right', fontsize=10)
ax.axvline(50, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
fig.tight_layout()
plt.savefig('fig8_lsa_share.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.9 Broadband Subscriber Growth

In [ ]:
bb_major = bb[bb['operator'].isin(['Reliance Jio','Bharti Airtel','Vodafone Idea','BSNL'])].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: line chart per operator
for op in ['Reliance Jio', 'Bharti Airtel', 'Vodafone Idea', 'BSNL']:
    d = bb_major[bb_major['operator'] == op].sort_values('month')
    if len(d):
        axes[0].plot(d['month'], d['broadband_subscribers_million'],
                     marker='o', linewidth=2.2, markersize=5,
                     label=op, color=OPERATOR_COLORS[op])
axes[0].set_title('Broadband Subscribers by Operator (Millions)', fontweight='bold')
axes[0].set_ylabel('Broadband Subscribers (M)')
axes[0].legend(fontsize=9)

# Right: national total
ns_bb = ns.dropna(subset=['broadband_total_million']).sort_values('month')
axes[1].fill_between(ns_bb['month'], ns_bb['broadband_total_million'],
                     alpha=0.15, color='#7c3aed')
axes[1].plot(ns_bb['month'], ns_bb['broadband_total_million'],
             marker='o', color='#7c3aed', linewidth=2.5)
axes[1].set_title('Total National Broadband Subscribers (Millions)', fontweight='bold')
axes[1].set_ylabel('Broadband Subscribers (M)')

fig.suptitle('Broadband Penetration Analysis', fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
plt.savefig('fig9_broadband.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.10 MNP (Mobile Number Portability) Trend

In [ ]:
ns_mnp = ns.dropna(subset=['mnp_requests_million']).sort_values('month')

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(ns_mnp['month'], ns_mnp['mnp_requests_million'],
                alpha=0.15, color='#d97706')
ax.plot(ns_mnp['month'], ns_mnp['mnp_requests_million'],
        marker='o', color='#d97706', linewidth=2.5, markersize=7)

for _, row in ns_mnp.iterrows():
    ax.annotate(f"{row['mnp_requests_million']:.2f}M",
                xy=(row['month'], row['mnp_requests_million']),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=8, color='#92400e')

ax.set_title('Monthly MNP (Mobile Number Portability) Requests (Millions)',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Month')
ax.set_ylabel('MNP Requests (Millions)')
fig.tight_layout()
plt.savefig('fig10_mnp.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.11 Urban vs Rural Telephone Penetration

In [ ]:
ns_ur = ns.dropna(subset=['urban_telephone_total_million','rural_telephone_total_million'])\
          .sort_values('month')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(ns_ur['month'], ns_ur['urban_telephone_total_million'],
        marker='s', linewidth=2.2, color='#1a3a5c', label='Urban')
ax.plot(ns_ur['month'], ns_ur['rural_telephone_total_million'],
        marker='^', linewidth=2.2, color='#16a34a', label='Rural')
ax.fill_between(ns_ur['month'],
                ns_ur['urban_telephone_total_million'],
                ns_ur['rural_telephone_total_million'],
                alpha=0.08, color='gray', label='Urban-Rural gap')

ax.set_title('Urban vs Rural Telephone Subscribers (Millions)',
             fontsize=13, fontweight='bold', pad=14)
ax.set_ylabel('Subscribers (Millions)')
ax.legend(fontsize=10)
fig.tight_layout()
plt.savefig('fig11_urban_rural.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.12 Correlation Heatmap — National Metrics

In [ ]:
ns_corr = ns[['broadband_total_million','telephone_total_million',
              'urban_telephone_total_million','rural_telephone_total_million',
              'mnp_requests_million','active_wireless_million']].dropna()

fig, ax = plt.subplots(figsize=(8, 6))
corr = ns_corr.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, square=True,
            xticklabels=['BB', 'Tel', 'Urban', 'Rural', 'MNP', 'Active'],
            yticklabels=['BB', 'Tel', 'Urban', 'Rural', 'MNP', 'Active'])
ax.set_title('Correlation Matrix — National Telecom Indicators',
             fontsize=13, fontweight='bold', pad=14)
fig.tight_layout()
plt.savefig('fig12_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Key Metrics & Statistical Analysis

In [ ]:
# ── 6.1 CAGR for each major operator ─────────────────────────────────────────
print('=== SUBSCRIBER GROWTH ANALYSIS (Jan 2025 → Aug 2026) ===')
first_month = nom['month'].min()
last_month  = nom['month'].max()
months_diff = (last_month.year - first_month.year) * 12 + (last_month.month - first_month.month)

for op in MAJOR_OPS:
    df_op = nom[nom['operator'] == op].sort_values('month')
    v0 = df_op.iloc[0]['subs_millions']
    v1 = df_op.iloc[-1]['subs_millions']
    change = v1 - v0
    pct    = (v1 / v0 - 1) * 100
    print(f"  {op:<25} : {v0:>7.1f}M → {v1:>7.1f}M  (Δ {change:+.1f}M / {pct:+.1f}%)")

In [ ]:
# ── 6.2 Market share shift (pp) ───────────────────────────────────────────────
print('=== MARKET SHARE SHIFT (Jan 2025 → Aug 2026) ===')
ms_first = nom[nom['month'] == first_month].set_index('operator')['market_share_pct']
ms_last  = nom[nom['month'] == last_month ].set_index('operator')['market_share_pct']
ms_shift = (ms_last - ms_first).sort_values(ascending=False)
for op, shift in ms_shift.items():
    print(f"  {op:<25} : {ms_first.get(op, 0):5.2f}% → {ms_last.get(op, 0):5.2f}%  ({shift:+.2f} pp)")

In [ ]:
# ── 6.3 LSA Leader Analysis ───────────────────────────────────────────────────
print('=== LSA MARKET LEADER — JAN 2025 ===')
lsa_j25_leader = (
    lsa_clean[lsa_clean['month'] == '2025-01-01']
    .groupby(['lsa','operator'])['subs_millions'].sum()
    .reset_index()
)
lsa_j25_leader['lsa_total'] = lsa_j25_leader.groupby('lsa')['subs_millions'].transform('sum')
lsa_j25_leader['share'] = lsa_j25_leader['subs_millions'] / lsa_j25_leader['lsa_total'] * 100

leader = (
    lsa_j25_leader.sort_values('subs_millions', ascending=False)
    .groupby('lsa').first()
    .reset_index()[['lsa','operator','subs_millions','share']]
    .rename(columns={'operator':'leader','subs_millions':'leader_subs_M','share':'leader_share_pct'})
    .sort_values('leader_share_pct', ascending=False)
)
print(leader.to_string(index=False))

In [ ]:
# ── 6.4 MNP Growth Statistics ─────────────────────────────────────────────────
ns_mnp_s = ns.dropna(subset=['mnp_requests_million']).sort_values('month')
mnp_min = ns_mnp_s['mnp_requests_million'].min()
mnp_max = ns_mnp_s['mnp_requests_million'].max()
mnp_growth = (mnp_max / mnp_min - 1) * 100

print(f'MNP Min  : {mnp_min:.2f}M ({ns_mnp_s.loc[ns_mnp_s["mnp_requests_million"].idxmin(), "month"].strftime("%b %Y")})')
print(f'MNP Max  : {mnp_max:.2f}M ({ns_mnp_s.loc[ns_mnp_s["mnp_requests_million"].idxmax(), "month"].strftime("%b %Y")})')
print(f'MNP Growth (min→max): {mnp_growth:.1f}%')

---
## 7. Key Findings & Business Insights

In [ ]:
insights = [
    ("1", "Airtel's Surge",
     "Bharti Airtel added ~104M subscribers (+27%) between Jan 2025 and Aug 2026, "
     "closing the gap with Jio from 78.8M to just 16.8M — the biggest market-share "
     "gain of any operator in the period."),
    ("2", "Jio's Market Share Erosion",
     "Despite adding 42.6M subscribers (+9.2%), Jio's market share fell from "
     "40.46% → 39.31% as Airtel's growth rate was 3× faster."),
    ("3", "Vi's Structural Decline & Stabilisation",
     "Vodafone Idea lost 6.3M subscribers (−3.1%) through 2025 but showed signs of "
     "stabilisation in 2026 post-fundraising. Share fell from 17.9% → 15.4%."),
    ("4", "BSNL's Government-Backed Revival",
     "BSNL reversed years of subscriber losses, gaining 2.1M net subs (+2.3%) with "
     "a notable +1.39M surge in Aug 2025 linked to government 4G rollout."),
    ("5", "India Crossed 1 Billion Broadband Subscribers",
     "Total broadband reached 999.8M (Oct 2025) → 1,101.2M (Aug 2026), "
     "a gain of 121.5M in 14 months (+12.4%). Jio leads with ~51% broadband share."),
    ("6", "Rising MNP Signals Competitive Churn",
     "Monthly MNP requests grew from 12.06M → 15.83M (+31.3%), indicating "
     "increasing consumer willingness to switch operators as pricing competition intensifies."),
    ("7", "Urban-Rural Digital Divide Widening",
     "Urban telephone subscribers grew +18.1% vs rural +3.7% (Aug 2025–Aug 2026). "
     "Rural share of total fell from 44.2% → 41.0%."),
    ("8", "Regional Battleground",
     "Airtel leads 5/23 LSAs (North East 50.3%, Assam 45.9%, Andhra Pradesh 41.0%). "
     "Jio dominates all remaining LSAs, strongest in Madhya Pradesh (55.7%) and Kolkata (47.4%).")
]

print('=== KEY FINDINGS ===')
for num, title, text in insights:
    print(f'\n[{num}] {title}')
    print(f'    {text}')

---
## 8. Export Cleaned Data

In [ ]:
# Export cleaned datasets for reproducibility
lsa_clean.to_csv('cleaned_lsa_wireless.csv', index=False)
nom.to_csv('cleaned_national_operator_monthly.csv', index=False)
ns.to_csv('cleaned_national_summary.csv', index=False)
bb.to_csv('cleaned_broadband_top5.csv', index=False)

print('✅ Exported:')
print('   cleaned_lsa_wireless.csv')
print('   cleaned_national_operator_monthly.csv')
print('   cleaned_national_summary.csv')
print('   cleaned_broadband_top5.csv')

---
## 9. Summary Dashboard Plot

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('TRAI Telecom Analytics — Summary Dashboard', fontsize=16, fontweight='bold', y=0.98)

ax1 = fig.add_subplot(2, 3, 1)
ax2 = fig.add_subplot(2, 3, 2)
ax3 = fig.add_subplot(2, 3, 3)
ax4 = fig.add_subplot(2, 3, 4)
ax5 = fig.add_subplot(2, 3, 5)
ax6 = fig.add_subplot(2, 3, 6)

# 1. National total growth
ax1.plot(monthly_total['month'], monthly_total['total_subs_millions'],
         color='steelblue', linewidth=2, marker='o', markersize=4)
ax1.set_title('National Subscriber Growth', fontweight='bold', fontsize=11)
ax1.set_ylabel('Subs (M)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}M'))

# 2. Market share pie
lt = nom[nom['month'] == latest_month].sort_values('market_share_pct', ascending=False)
ax2.pie(lt['market_share_pct'], labels=lt['operator'],
        colors=[OPERATOR_COLORS.get(o,'#999') for o in lt['operator']],
        autopct='%1.0f%%', startangle=140, pctdistance=0.8,
        wedgeprops=dict(edgecolor='white'))
ax2.set_title(f'Market Share ({latest_month.strftime("%b %Y")})', fontweight='bold', fontsize=11)

# 3. Airtel vs Jio gap
ax3.plot(gap['month'], gap['gap_millions'], color='purple', linewidth=2, marker='o', markersize=4)
ax3.axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax3.fill_between(gap['month'], gap['gap_millions'], alpha=0.15, color='purple')
ax3.set_title('Jio–Airtel Gap (M)', fontweight='bold', fontsize=11)
ax3.set_ylabel('Gap (M)')

# 4. MNP
ns_mnp2 = ns.dropna(subset=['mnp_requests_million']).sort_values('month')
ax4.bar(ns_mnp2['month'], ns_mnp2['mnp_requests_million'],
        color='#d97706', alpha=0.8, width=20)
ax4.set_title('MNP Requests (M)', fontweight='bold', fontsize=11)
ax4.set_ylabel('MNP (M)')

# 5. Broadband
ax5.fill_between(ns_bb['month'], ns_bb['broadband_total_million'],
                 alpha=0.2, color='#7c3aed')
ax5.plot(ns_bb['month'], ns_bb['broadband_total_million'],
         color='#7c3aed', linewidth=2, marker='o', markersize=4)
ax5.set_title('Broadband Subscribers (M)', fontweight='bold', fontsize=11)
ax5.set_ylabel('Subs (M)')

# 6. Top 5 LSAs bar
top5_lsa = lsa_totals_jan25.head(5)
ax6.barh(top5_lsa['lsa'], top5_lsa['total_subs_millions'],
         color=['#3b82d4','#e65100','#2e7d32','#1565c0','#880e4f'])
ax6.set_title('Top 5 LSAs by Subscribers — Jan 2025', fontweight='bold', fontsize=11)
ax6.set_xlabel('Subs (M)')
ax6.invert_yaxis()

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('fig13_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Summary dashboard saved.')

---
## 10. Conclusions

| # | Finding | Implication |
|---|---------|-------------|
| 1 | Airtel grew 27% vs Jio's 9.2% | Indian market shifting to a duopoly |
| 2 | India hit 1.1B broadband subs (Aug 2026) | One of world's largest broadband markets |
| 3 | MNP requests up 31.3% | Intensifying price competition; consumers more willing to churn |
| 4 | Vi stabilising in 2026 | Post-fundraising investment beginning to show |
| 5 | BSNL reversed decline | Government 4G rollout having measurable effect |
| 6 | Urban growing 5× faster than rural | Digital divide requires targeted rural policy |
| 7 | Jio leads 10/23 LSAs, Airtel leads North/South | Regional competition remains fierce |

**Data Source:** TRAI Press Releases — Telecom Subscription Data (Feb 2025 – Aug 2026)  
**Limitation:** 5 months of LSA-level data (Sep 2025–Jan 2026) unavailable due to source gaps.